# 08 - Real-gap candidate reconstructions

Loads the precomputed real-gap candidate output table and shows both
candidate methods (TS-ICL satellite-proxy and the engineered hybrid
pipeline) side by side. Fully executable on the public data included in
this repository.

**Real gaps have no withheld ground truth.** The outputs shown here are
plausible candidate values, not validation evidence -- see
`docs/evidence_hierarchy.md` for how to interpret them responsibly.


In [ ]:
import pandas as pd

real_gaps = pd.read_csv(
    "../results_public/chlorophyll/chlorophyll_real_gap_candidate_outputs.csv",
    parse_dates=["start_date", "end_date"],
)
real_gaps.head()


## Reproducing the inventory and assembly (not just reading the frozen file)

Everything above and below this cell reads the already-assembled, frozen
`chlorophyll_real_gap_candidate_outputs.csv`. The cells in this section
instead run the actual deterministic code
(`experiments.chlorophyll.real_gap_inventory`,
`assemble_real_gap_candidates`, `select_real_gap_reconstruction`) that
detects real gaps from the daily target and joins the per-method candidate
files -- this is the code path the earlier publication audit found missing
from the public repository. No TS-ICL call, no model fit: pure detection
and joining over already-frozen inputs, so this runs in well under a
second.

In [ ]:
import sys

sys.path.insert(0, "..")

from experiments.chlorophyll import real_gap_inventory as ri
from experiments.chlorophyll import select_real_gap_reconstruction as sr
from experiments.chlorophyll.assemble_real_gap_candidates import run_assembly

target_df = pd.read_csv(
    "../data_public/chlorophyll/chlorophyll_daily_target.csv", parse_dates=["date"]
).set_index("date").sort_index()

inventory = ri.detect_real_gaps(target_df)
print(f"Detected {len(inventory)} real gaps from the daily target's eligibility column alone "
      f"(no candidate file was read to find these boundaries).")
inventory.head(3)


## Two different kinds of candidate: method-selected vs. independent

The two methods shown throughout this notebook are not symmetric:

- **`engineered_hybrid`** is a **method-selected candidate**: for each real
  gap, a deterministic rule ("Rule D") assigns exactly one component method
  by gap length -- GP (L1-3), a state-space Kalman smoother (L4-29), or a
  gap-edge residual model (L>=30) -- and that component's output becomes
  the reported value. `select_real_gap_reconstruction.route_real_gaps`
  below reproduces this routing decision exactly (not a new fit -- a
  lookup over which method *would have been* used, verified against the
  already-frozen output).
- **`tsicl_satellite_proxy`** is a single method applied uniformly to every
  gap -- no per-gap routing.

Neither is "the" selected final reconstruction across *both* methods --
per this project's benchmark-first framing, no single real-gap series is
presented as uniquely correct. They remain two independent, differently-
constructed candidates, shown side by side.

In [ ]:
routed = sr.route_real_gaps(inventory)
excluded = routed[routed["assigned_method"].isna()]
print(f"Rule D assigns a method to {routed['assigned_method'].notna().sum()} of {len(routed)} real gaps.")
print(f"Excluded (no post-edge context available): {excluded['gap_id'].tolist()}")
routed[["gap_id", "length_days", "post_edge_available", "assigned_method"]].head(10)


## Running the deterministic assembly

Joins the two frozen per-method candidate files with the just-detected
inventory, validates the join (unique rows, exact gap-day coverage, dates
within the declared gap window, finite predictions, ordered quantiles), and
writes to `build/chlorophyll/real_gap_candidates/` -- never overwrites
`results_public/`.

In [ ]:
from pathlib import Path

rc = run_assembly(Path("../build/chlorophyll/real_gap_candidates"))
print(f"\nExit code: {rc} (0 = validation passed)")


## Gap-length distribution of real gaps

In [ ]:
real_gaps["length_days"].describe()


## Reminder

- Artificial-gap validation (notebooks 02, 03, 07) supports ranking methods
  -- gaps there have a withheld, known true value. Every real gap shown in
  this notebook does not; every value above is a **candidate**, never an
  observation.
- `engineered_hybrid` is a method-*selected* candidate (Rule D, length-
  routed); `tsicl_satellite_proxy` is a single method applied uniformly.
  Neither is promoted as the one correct reconstruction across both.
- The 256-day gap is far outside the validated gap-length envelope
  (maximum validated length: 60 days) and should be treated as illustrative
  only -- flagged `scenario_only_256day` throughout.
- `experiments/chlorophyll/real_gap_contract.py` is the single source of
  truth for which published artifact is which kind of evidence -- consult
  it (`REAL_GAP_ARTIFACTS`) before citing any real-gap number elsewhere.


In [ ]:
comparison = real_gaps[[
    "gap_id", "start_date", "length_days",
    "tsicl_satellite_proxy_mean_pred_chl",
    "engineered_hybrid_mean_reconstructed_chl",
    "extrapolation_beyond_validation",
    "scenario_only_256day",
]]
comparison.head(15)


## Where the two methods diverge

Large divergence between the two candidate methods on a given real gap is a
useful (informal) red flag -- it suggests at least one of them is
extrapolating into an unfamiliar regime for that gap, even though we can't
say which one (no ground truth exists to check).


In [ ]:
comparison = comparison.copy()
comparison["abs_divergence"] = (
    comparison["tsicl_satellite_proxy_mean_pred_chl"]
    - comparison["engineered_hybrid_mean_reconstructed_chl"]
).abs()
comparison.sort_values("abs_divergence", ascending=False).head(10)


## The 256-day scenario gap

In [ ]:
scenario_rows = real_gaps[real_gaps["scenario_only_256day"] == True]
scenario_rows[["gap_id", "start_date", "end_date", "length_days", "note_256day_scenario"]]


## Reminder

- Artificial-gap validation (notebooks 02, 03, 07) supports ranking methods.
- Real-gap outputs shown here are candidate values only -- useful for
  plotting a continuous series or rough magnitude checks, not for claiming
  one method is more accurate than another.
- The 256-day gap is far outside the validated gap-length envelope
  (maximum validated length: 60 days) and should be treated as illustrative
  only.
